In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml import command, load_component
from azure.ai.ml.dsl import pipeline
from azure.ai.ml import Input, Output
from azureml.core import Workspace, Dataset, Datastore

subscription_id = '813ac1cb-7b8e-45ab-9b3c-8510079c761a'
resource_group = 'MIBLab_Official'
workspace_name = 'miblab-official-ml'

workspace = Workspace(subscription_id, resource_group, workspace_name)
datastore = Datastore.get(workspace, "eegfoundationmodeldata")

# Get a handle to the workspace
ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name,
)

cluster_name = "miblab-FX24mds-cpu"

### HBN

In [ ]:
processingHBN_component = load_component(source="./GraphCreationJobHBN.yml")

data_dir = Output(
	type="uri_folder",
	mode='rw_mount',
	path="azureml://subscriptions/813ac1cb-7b8e-45ab-9b3c-8510079c761a/resourcegroups/MIBLab_Official/workspaces/miblab-official-ml/datastores/eegfoundationmodeldata/paths/HBN-Processed"
)

# Define a pipeline with component
@pipeline(default_compute=cluster_name)
def processingHBN():
	processing = processingHBN_component()
	return {
        "output_dir": processing.outputs.output_dir
    }

processingHBN_job = processingHBN()
processingHBN_job.outputs.output_dir = data_dir

# submit job to workspace
processingHBN_job = ml_client.jobs.create_or_update(
	processingHBN_job,
    experiment_name="EEGFoundationModel",
)

processingHBN_job

### TUH

In [ ]:
processingTUH_component = load_component(source="./GraphCreationJobTUH.yml")

data_dir = Output(
	type="uri_folder",
	mode='rw_mount',
	path="azureml://subscriptions/813ac1cb-7b8e-45ab-9b3c-8510079c761a/resourcegroups/MIBLab_Official/workspaces/miblab-official-ml/datastores/eegfoundationmodeldata/paths/TUH-Processed"
)

# Define a pipeline with component
@pipeline(default_compute=cluster_name)
def processingTUH():
	processing = processingTUH_component()
	return {
        "output_dir": processing.outputs.output_dir
    }

processingTUH_job = processingTUH()
processingTUH_job.outputs.output_dir = data_dir

# submit job to workspace
processingTUH_job = ml_client.jobs.create_or_update(
	processingTUH_job,
    experiment_name="EEGFoundationModel",
)

processingTUH_job